# Clean up a noisy recording with Inworld Voice Isolator

This notebook takes a noisy speech recording, removes what is around the voice, and lets you hear the result next to the original. It then does the same for a recording of your own.

You will:

1. Install the package and check this machine.
2. Load the offline model (its weights download on first use).
3. Run the bundled noisy sample through it.
4. Listen to the original and the cleaned audio side by side.
5. Swap in your own recording and save the output.

**What you need**

- Python 3.10 or later and PyTorch 2.1 or later.
- About 260 MB of disk for the offline model.
- A GPU is optional. The model is tested on NVIDIA A100, H100, B200 and B300 GPUs and on CPU; smaller GPUs are not tested yet. On an Apple-silicon Mac it uses the Metal GPU with PyTorch 2.6 or later. On CPU it works, more slowly.

This notebook covers offline cleanup of recorded audio. For live audio, see the streaming section of the [README](https://github.com/inworld-ai/voice-isolator#streaming).

## 1. Install and check the machine

Inside a clone of the repository this installs the checkout. Anywhere else (Colab, for example) it installs the release from PyPI. If the package is already installed in this environment, the cell leaves it alone.

In [ ]:
import pathlib

REPO = pathlib.Path.cwd().parent
IN_CLONE = (REPO / "src" / "inworld" / "voice_isolation").is_dir()

try:
    from inworld import voice_isolation
    print(f"inworld-voice-isolation {voice_isolation.__version__} is already installed.")
except ImportError:
    if IN_CLONE:
        %pip install -q "{REPO}"
    else:
        %pip install -q inworld-voice-isolation

`voice-isolate-doctor` reports the Python and PyTorch versions and the device inference will use. It prints one line per check, starting with `OK`, `INFO`, `WARN` or `FAIL`; a `FAIL` line names what to fix. This quick form uses no network and loads no model.

In [ ]:
import sys

!"{sys.executable}" -m inworld.voice_isolation.doctor

## 2. Load the model

The first run downloads the offline model, `voice-isolator-v1` (256 MB), from [huggingface.co/inworld/voice-isolator](https://huggingface.co/inworld/voice-isolator) into the Hugging Face cache and verifies its checksum. No account or sign-in is needed. Later runs load it from the cache.

Code and weights are Apache-2.0.

In [ ]:
import time

import torch
from IPython.display import Audio, display

from inworld import voice_isolation

isolator = voice_isolation.VoiceIsolator("inworld/voice-isolator")
print(f"Model loaded on: {isolator.device}")

## 3. Clean the bundled sample

The sample is 9 seconds of read speech (from LibriSpeech, CC BY 4.0) mixed with a low electrical hum and three sharp taps.

`isolate` returns the voice at the input's sample rate, length and level. Subtracting it from the input gives what the model removed.

In [ ]:
SAMPLE_URL = ("https://raw.githubusercontent.com/inworld-ai/voice-isolator/"
              "main/demo/examples/sample_15.wav")

sample = REPO / "demo" / "examples" / "sample_15.wav"
if not sample.is_file():
    import urllib.request
    sample = pathlib.Path("sample_15.wav")
    urllib.request.urlretrieve(SAMPLE_URL, sample)


def clean(path):
    """Returns (original, voice, background, sample_rate) for one file."""
    wav, sample_rate = voice_isolation.load_audio(str(path))
    start = time.perf_counter()
    voice = isolator.isolate(wav, sample_rate).cpu()
    elapsed = time.perf_counter() - start
    seconds = len(wav) / sample_rate
    print(f"{pathlib.Path(path).name}: {seconds:.1f} s of audio cleaned in "
          f"{elapsed:.1f} s ({seconds / elapsed:.1f}x real time)")
    return wav, voice, wav - voice, sample_rate


original, voice, background, sample_rate = clean(sample)

The first call on a GPU includes one-time warm-up, so run the cell again for a representative speed.

## 4. Listen side by side

The players keep the true levels (no normalization), so loudness is comparable between them.

What to listen for:

- **Noise.** Are the hum and the taps gone, including in the pauses between words?
- **Speech.** Does the voice still sound natural? Check consonants and word endings. Breaths and other non-verbal sounds are kept on purpose.
- **Removed background.** It should hold the noise and little or no voice. Voice you hear here is voice the model took out.

In [ ]:
def play(label, wav):
    """Shows a player at the waveform's own level, without normalization."""
    print(label)
    display(Audio(wav.clamp(-1, 1).numpy(), rate=sample_rate, normalize=False))


play("Original", original)
play("Cleaned voice", voice)
play("Removed background", background)

## 5. Try your own recording

Set `MY_RECORDING` to a file on this machine and run the cell. In Colab, upload the file with the folder icon on the left first.

- **Formats:** WAV, FLAC, OGG and MP3 (anything the `soundfile` library reads). Convert other formats first, for example `ffmpeg -i input.m4a input.wav`.
- **Sample rate and channels:** any sample rate works. Stereo is mixed down to mono, and the output is mono.
- **Length:** long files are processed in 8-second chunks that overlap by 2 seconds and are crossfaded.

The cell writes two float32 WAVs beside your file: `<name>_clean.wav` and `<name>_clean_background.wav`. Your audio stays on this machine.

In [ ]:
MY_RECORDING = ""  # for example "recordings/call.wav"

if MY_RECORDING:
    source = pathlib.Path(MY_RECORDING).expanduser()
    original, voice, background, sample_rate = clean(source)

    voice_path = source.with_name(f"{source.stem}_clean.wav")
    background_path = source.with_name(f"{source.stem}_clean_background.wav")
    voice_isolation.save_audio(str(voice_path), voice, sample_rate,
                               subtype="FLOAT")
    voice_isolation.save_audio(str(background_path), background, sample_rate,
                               subtype="FLOAT")
    print(f"Saved {voice_path} and {background_path}")

    play("Original", original)
    play("Cleaned voice", voice)
    play("Removed background", background)
else:
    print("Set MY_RECORDING to the path of an audio file, then run this cell.")

## Use it in your own code

The whole pipeline is a few lines:

```python
from inworld import voice_isolation

isolator = voice_isolation.VoiceIsolator("inworld/voice-isolator")  # load once, reuse
wav, sample_rate = voice_isolation.load_audio("noisy.wav")
voice = isolator.isolate(wav, sample_rate)
voice_isolation.save_audio("clean.wav", voice, sample_rate)
```

Or from a terminal, for one file or a whole folder:

```bash
voice-isolate noisy.wav clean.wav
voice-isolate recordings/ cleaned/
```

**Cleaning audio before speech-to-text?** Ask for the rate your STT model expects, and the voice comes back resampled:

```python
voice_16k = isolator.isolate(wav, sample_rate, output_sample_rate=16000)
```

Where to go next:

- [README](https://github.com/inworld-ai/voice-isolator#quick-start): CLI options and the streaming model for live audio.
- [Interactive demo](https://github.com/inworld-ai/voice-isolator/blob/main/demo/README.md): a browser page with file upload and live microphone.
- [Model card](https://github.com/inworld-ai/voice-isolator/blob/main/MODEL_CARD.md): what the model does and does not do, benchmark results and training data.